In [11]:
import sys
import os

venv_site_packages = "/Users/gk/Downloads/razorpay/.venv/lib/python3.11/site-packages"
if venv_site_packages not in sys.path:
    sys.path.insert(0, venv_site_packages)

In [12]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python path: {sys.path}")

import fraud_spike_detector
import json

streaming_layer = fraud_spike_detector.PyStreamingLayer(
    alpha=0.05,
    cusum_threshold=4.0,
    cusum_drift=0.5
)

# Example: Process a batch of transactions
sample_transactions = [
    fraud_spike_detector.PyTransaction(
        id="tx_001",
        merchant_id="merchant_123",
        bin="411111",
        is_disputed=False,
        timestamp_ms=1725249600000
    ),
    fraud_spike_detector.PyTransaction(
        id="tx_002",
        merchant_id="merchant_123",
        bin="411111",
        is_disputed=True,
        timestamp_ms=1725249601000
    ),
]

# Process transactions and get alerts
alerts = streaming_layer.process_transaction_batch(sample_transactions)
print(f"Rust Streaming Engine loaded and running at C speeds!")
print(f"Processed {len(sample_transactions)} transactions")
print(f"Alerts detected: {len(alerts)}")

Processed 2 transactions
Alerts detected: 0


In [13]:
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# =====================================================================
# 1. DATASET GENERATION (In-Memory, Zero External File Dependencies)
# =====================================================================

def generate_fraud_dataset(
    n_samples: int = 50_000, fraud_rate: float = 0.02, seed: int = 42
):
    rng = np.random.default_rng(seed)
    n_fraud = int(n_samples * fraud_rate)
    n_clean = n_samples - n_fraud

    # Stage 1 Cheap Features
    amount_clean = rng.lognormal(mean=3.5, sigma=1.0, size=n_clean)
    velocity_clean = rng.poisson(lam=1.2, size=n_clean)
    hour_clean = rng.integers(0, 24, size=n_clean)
    risk_cat_clean = rng.choice([0, 1, 2], p=[0.7, 0.2, 0.1], size=n_clean)

    amount_fraud = rng.lognormal(mean=4.2, sigma=1.2, size=n_fraud)
    velocity_fraud = rng.poisson(lam=3.8, size=n_fraud)
    hour_fraud = rng.choice([0, 1, 2, 3, 4, 22, 23], size=n_fraud)
    risk_cat_fraud = rng.choice([0, 1, 2], p=[0.1, 0.3, 0.6], size=n_fraud)

    # Stage 2 Expensive Features
    device_shared_clean = rng.binomial(n=1, p=0.05, size=n_clean)
    ip_reputation_clean = rng.beta(a=8, b=2, size=n_clean)
    merchant_cb_rate_clean = rng.exponential(scale=0.005, size=n_clean)
    graph_risk_score_clean = rng.beta(a=2, b=8, size=n_clean)

    device_shared_fraud = rng.binomial(n=1, p=0.45, size=n_fraud)
    ip_reputation_fraud = rng.beta(a=2, b=5, size=n_fraud)
    merchant_cb_rate_fraud = rng.exponential(scale=0.04, size=n_fraud)
    graph_risk_score_fraud = rng.beta(a=6, b=3, size=n_fraud)

    df_clean = pd.DataFrame({
        "amount": amount_clean,
        "velocity_1h": velocity_clean,
        "hour_of_day": hour_clean,
        "merchant_risk_cat": risk_cat_clean,
        "device_shared_count": device_shared_clean,
        "ip_reputation_score": ip_reputation_clean,
        "merchant_historical_cb_rate": merchant_cb_rate_clean,
        "graph_cluster_risk": graph_risk_score_clean,
        "is_fraud": 0,
    })

    df_fraud_data = pd.DataFrame({
        "amount": amount_fraud,
        "velocity_1h": velocity_fraud,
        "hour_of_day": hour_fraud,
        "merchant_risk_cat": risk_cat_fraud,
        "device_shared_count": device_shared_fraud,
        "ip_reputation_score": ip_reputation_fraud,
        "merchant_historical_cb_rate": merchant_cb_rate_fraud,
        "graph_cluster_risk": graph_risk_score_fraud,
        "is_fraud": 1,
    })

    df = pd.concat([df_clean, df_fraud_data]).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    df["timestamp"] = pd.date_range(start="2026-08-01", periods=len(df), freq="1s")
    return df

# =====================================================================
# 2. MODEL CLASSES (Stage 1, Stage 2, Split Conformal)
# =====================================================================

class Stage1HotPath:
    def __init__(self, target_recall: float = 0.99):
        self.target_recall = target_recall
        self.scaler = StandardScaler()
        self.model = LogisticRegression(C=1.0, max_iter=500, class_weight="balanced")
        self.threshold = 0.5

    def fit(self, X: pd.DataFrame, y: pd.Series):
        X_scaled = self.scaler.fit_transform(X)
        self.model.fit(X_scaled, y)
        probs = self.model.predict_proba(X_scaled)[:, 1]

        thresholds = np.linspace(0.001, 0.999, 1000)
        valid_thresholds = []

        for th in thresholds:
            preds = (probs >= th).astype(int)
            true_positives = np.sum((preds == 1) & (y == 1))
            total_positives = np.sum(y == 1)
            recall = true_positives / total_positives if total_positives > 0 else 1.0

            if recall >= self.target_recall:
                valid_thresholds.append(th)

        self.threshold = max(valid_thresholds) if valid_thresholds else 0.01

    def predict_escalate(self, X: pd.DataFrame):
        X_scaled = self.scaler.transform(X)
        start_t = time.perf_counter()
        probs = self.model.predict_proba(X_scaled)[:, 1]
        latencies_ms = (time.perf_counter() - start_t) * 1000 / len(X)
        escalate_mask = probs >= self.threshold
        return escalate_mask, probs, latencies_ms

class Stage2WarmPath:
    def __init__(self, cost_ratio: float = 10.0):
        self.cost_ratio = cost_ratio
        self.model = None

    def fit(self, X: pd.DataFrame, y: pd.Series):
        train_data = lgb.Dataset(X, label=y)
        params = {
            "objective": "binary",
            "metric": "binary_logloss",
            "boosting_type": "gbdt",
            "scale_pos_weight": self.cost_ratio,
            "num_leaves": 31,
            "learning_rate": 0.05,
            "feature_fraction": 0.8,
            "verbose": -1,
            "seed": 42,
        }
        self.model = lgb.train(params, train_data, num_boost_round=150)

    def predict_proba(self, X: pd.DataFrame):
        start_t = time.perf_counter()
        probs = self.model.predict(X)
        latency_ms = (time.perf_counter() - start_t) * 1000 / max(len(X), 1)
        probs_2d = np.column_stack([1 - probs, probs])
        return probs_2d, latency_ms

class SplitConformalPredictor:
    def __init__(self, alpha: float = 0.05):
        self.alpha = alpha
        self.q_hat = None

    def calibrate(self, cal_probs: np.ndarray, cal_labels: np.ndarray):
        n = len(cal_labels)
        true_class_probs = cal_probs[np.arange(n), cal_labels]
        scores = 1.0 - true_class_probs
        quantile_val = np.ceil((n + 1) * (1 - self.alpha)) / n
        quantile_val = min(1.0, quantile_val)
        self.q_hat = np.quantile(scores, quantile_val, method="higher")

    def predict_set(self, test_probs: np.ndarray):
        if self.q_hat is None:
            raise ValueError("Predictor must be calibrated before inference.")

        sets = []
        for p in test_probs:
            p0, p1 = p[0], p[1]
            s0, s1 = 1.0 - p0, 1.0 - p1

            pred_set = set()
            if s0 <= self.q_hat:
                pred_set.add(0)
            if s1 <= self.q_hat:
                pred_set.add(1)

            if len(pred_set) == 0:
                pred_set.add(int(np.argmax(p)))
            sets.append(pred_set)

        return sets

    @staticmethod
    def evaluate_coverage(pred_sets, true_labels):
        covered = [y in p_set for y, p_set in zip(true_labels, pred_sets)]
        set_sizes = [len(s) for s in pred_sets]
        coverage = np.mean(covered)
        avg_set_size = np.mean(set_sizes)
        ambiguous_pct = np.mean([1 if len(s) > 1 else 0 for s in pred_sets])
        return coverage, avg_set_size, ambiguous_pct

# =====================================================================
# 3. FINANCIAL IMPACT ESTIMATOR (Layer 3 — Net Saved Margin)
# =====================================================================

def calculate_net_saved_margin(test_df, pred_sets, esc_mask_test, cost_per_review=15.0, chargeback_fee=25.0):
    """
    Net Saved Margin = (Fraud Amount Prevented) - (Manual Review Cost) - (Missed Fraud Chargeback Cost)
    """
    total_fraud_volume = test_df[test_df["is_fraud"] == 1]["amount"].sum()
    
    # Track actions based on conformal sets
    test_escalated = test_df[esc_mask_test].copy().reset_index(drop=True)
    
    review_costs = 0.0
    fraud_prevented = 0.0
    missed_fraud_costs = 0.0

    # Auto-cleared by Stage 1
    cleared_by_s1 = test_df[~esc_mask_test]
    s1_missed_fraud = cleared_by_s1[cleared_by_s1["is_fraud"] == 1]
    missed_fraud_costs += s1_missed_fraud["amount"].sum() + (len(s1_missed_fraud) * chargeback_fee)

    # Process Stage 2 set predictions
    for idx, p_set in enumerate(pred_sets):
        row = test_escalated.iloc[idx]
        is_actual_fraud = row["is_fraud"] == 1
        amt = row["amount"]

        if p_set == {1}: # Definite Fraud -> Blocked
            if is_actual_fraud:
                fraud_prevented += amt
        elif p_set == {0, 1}: # Ambiguous -> Routed to Manual Review
            review_costs += cost_per_review
            if is_actual_fraud:
                fraud_prevented += amt # Reviewer catches fraud
        elif p_set == {0}: # Definite Clean -> Auto-Approved
            if is_actual_fraud:
                missed_fraud_costs += amt + chargeback_fee

    net_saved_margin = fraud_prevented - review_costs - missed_fraud_costs
    return net_saved_margin, fraud_prevented, review_costs, missed_fraud_costs

# =====================================================================
# 4. PIPELINE RUNNER
# =====================================================================

print("==================================================")
print("   ML SCORING LAYER — TWO-STAGE CASCADE ENGINE    ")
print("==================================================\n")

STAGE1_FEATURES = ["amount", "velocity_1h", "hour_of_day", "merchant_risk_cat"]
STAGE2_FEATURES = STAGE1_FEATURES + ["device_shared_count", "ip_reputation_score", "merchant_historical_cb_rate", "graph_cluster_risk"]
TARGET = "is_fraud"

df_fraud = generate_fraud_dataset(n_samples=50_000, fraud_rate=0.02)
train_idx, cal_idx = int(0.60 * len(df_fraud)), int(0.80 * len(df_fraud))

train_df = df_fraud.iloc[:train_idx].copy()
cal_df = df_fraud.iloc[train_idx:cal_idx].copy()
test_df = df_fraud.iloc[cal_idx:].copy()

# Fit Stage 1
stage1 = Stage1HotPath(target_recall=0.99)
stage1.fit(train_df[STAGE1_FEATURES], train_df[TARGET])

# Fit Stage 2
esc_mask_train, _, _ = stage1.predict_escalate(train_df[STAGE1_FEATURES])
stage2 = Stage2WarmPath(cost_ratio=10.0)
stage2.fit(train_df[esc_mask_train][STAGE2_FEATURES], train_df[esc_mask_train][TARGET])

# Calibrate Conformal Layer
esc_mask_cal, _, _ = stage1.predict_escalate(cal_df[STAGE1_FEATURES])
cal_stage2_probs, _ = stage2.predict_proba(cal_df[esc_mask_cal][STAGE2_FEATURES])
conformal = SplitConformalPredictor(alpha=0.05)
conformal.calibrate(cal_stage2_probs, cal_df[esc_mask_cal][TARGET].values)

# Test Inference
esc_mask_test, _, s1_latency_ms = stage1.predict_escalate(test_df[STAGE1_FEATURES])
s2_probs, s2_latency_ms = stage2.predict_proba(test_df[esc_mask_test][STAGE2_FEATURES])
pred_sets = conformal.predict_set(s2_probs)

coverage, avg_set_size, ambig_rate = conformal.evaluate_coverage(pred_sets, test_df[esc_mask_test][TARGET].values)

# Financial Impact Analysis
net_margin, caught_amt, review_costs, loss_amt = calculate_net_saved_margin(
    test_df, pred_sets, esc_mask_test
)

print("================ SUMMARY METRICS ================")
print(f"  1. [Throughput] {100*(1 - np.mean(esc_mask_test)):.1f}% of traffic auto-cleared by Stage 1.")
print(f"  2. [Rigor] Conformal wrapper yields a {100*coverage:.1f}% empirical coverage set (Target: ≥95.0%).")
print(f"  3. [Actionability] Only {100*ambig_rate*np.mean(esc_mask_test):.1f}% of total traffic routed to Human Review.")
print(f"  4. [Fraud Prevented] ${caught_amt:,.2f} caught out of test set.")
print(f"  5. [Net Saved Margin] ${net_margin:,.2f} net financial impact.")
print("==================================================")

   ML SCORING LAYER — TWO-STAGE CASCADE ENGINE    



/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


================ SUMMARY METRICS ================
  1. [Throughput] 30.0% of traffic auto-cleared by Stage 1.
  2. [Rigor] Conformal wrapper yields a 99.9% empirical coverage set (Target: ≥95.0%).
  3. [Actionability] Only 0.0% of total traffic routed to Human Review.
  4. [Fraud Prevented] $28,198.31 caught out of test set.
  5. [Net Saved Margin] $27,135.19 net financial impact.
